# Customer Churn Prediction — Exploratory Data Analysis

**Dataset:** IBM Telco Customer Churn (7,043 customers, 21 features)  
**Goal:** Understand the data, find patterns related to churn, and prepare insights for feature engineering.

---
**Notebook structure:**
1. Load data & first look
2. Data quality checks
3. Target variable — class imbalance
4. Numerical features
5. Categorical features
6. Churn rate by feature
7. Correlation analysis
8. Key findings summary

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

# consistent style for all plots
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120
CHURN_PALETTE = {'No': '#4C72B0', 'Yes': '#DD8452'}

DATA_PATH = '../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv'
print('Libraries loaded.')

## 1. Load Data & First Look

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f'Rows: {df_raw.shape[0]:,}  |  Columns: {df_raw.shape[1]}')
df_raw.head()

In [ ]:
# Data types and non-null counts
df_raw.info()

In [ ]:
# Basic statistics for numeric columns
df_raw.describe()

## 2. Data Quality Checks

In [ ]:
# Check for missing values
missing = df_raw.isnull().sum()
print('Columns with missing values:')
print(missing[missing > 0] if missing.any() else 'None found at first glance.')

In [ ]:
# TotalCharges is stored as string — convert and find hidden blanks
# Customers with tenure=0 have a blank string instead of 0
df_raw['TotalCharges'] = pd.to_numeric(df_raw['TotalCharges'], errors='coerce')

hidden_nulls = df_raw['TotalCharges'].isnull().sum()
print(f'Hidden nulls in TotalCharges after conversion: {hidden_nulls}')
df_raw[df_raw['TotalCharges'].isnull()][['customerID', 'tenure', 'MonthlyCharges', 'TotalCharges']]

In [ ]:
# These are new customers (tenure=0) — fill with 0
df_raw['TotalCharges'] = df_raw['TotalCharges'].fillna(0)

# Encode target: Yes -> 1, No -> 0
df_raw['Churn_binary'] = (df_raw['Churn'] == 'Yes').astype(int)

# Check duplicates
print(f'Duplicate rows: {df_raw.duplicated().sum()}')
print(f'Duplicate customerIDs: {df_raw["customerID"].duplicated().sum()}')
print('Data is clean and ready.')

In [ ]:
# Split columns by type for easier reference later
NUM_COLS = ['tenure', 'MonthlyCharges', 'TotalCharges']

CAT_COLS = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod'
]

print(f'Numeric features : {NUM_COLS}')
print(f'Categorical features: {len(CAT_COLS)} columns')

## 3. Target Variable — Class Imbalance

In [ ]:
churn_counts = df_raw['Churn'].value_counts()
churn_pct    = df_raw['Churn'].value_counts(normalize=True) * 100

print('Churn distribution:')
for label in ['No', 'Yes']:
    print(f'  {label}: {churn_counts[label]:,} ({churn_pct[label]:.1f}%)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Count plot
sns.countplot(data=df_raw, x='Churn', palette=CHURN_PALETTE, ax=axes[0])
for bar in axes[0].patches:
    axes[0].text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 30,
        f'{int(bar.get_height()):,}',
        ha='center', fontsize=11
    )
axes[0].set_title('Churn Count')
axes[0].set_xlabel('')

# Pie chart
axes[1].pie(
    churn_counts,
    labels=['No Churn', 'Churned'],
    autopct='%1.1f%%',
    colors=[CHURN_PALETTE['No'], CHURN_PALETTE['Yes']],
    startangle=90,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2}
)
axes[1].set_title('Churn Split')

plt.suptitle('Target Variable: Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/01_churn_distribution.png', dpi=150)
plt.show()

# The dataset is imbalanced (~73% No, ~27% Yes).
# We must use AUC-PR as our main metric — not accuracy.

## 4. Numerical Features

In [ ]:
df_raw[NUM_COLS].describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

for i, col in enumerate(NUM_COLS):
    # Top row: overall distribution
    sns.histplot(df_raw[col], bins=40, ax=axes[0, i], color='steelblue', kde=True)
    axes[0, i].set_title(f'{col} — Overall Distribution')

    # Bottom row: split by churn
    for churn_val, color in CHURN_PALETTE.items():
        subset = df_raw[df_raw['Churn'] == churn_val][col]
        axes[1, i].hist(subset, bins=40, alpha=0.6, color=color, label=churn_val, density=True)
    axes[1, i].set_title(f'{col} — By Churn')
    axes[1, i].legend(title='Churn')

plt.suptitle('Numerical Features Distribution', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/02_numerical_distributions.png', dpi=150)
plt.show()

In [ ]:
# Box plots to spot outliers clearly
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for i, col in enumerate(NUM_COLS):
    sns.boxplot(
        data=df_raw, x='Churn', y=col,
        palette=CHURN_PALETTE, ax=axes[i]
    )
    axes[i].set_title(f'{col} by Churn')
    axes[i].set_xlabel('')

plt.suptitle('Numerical Features — Churn vs No Churn', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/03_numerical_boxplots.png', dpi=150)
plt.show()

# Key observations:
# - Churned customers have much shorter tenure (new customers leave more).
# - Churned customers pay higher monthly charges.
# - Churned customers have lower total charges (because they leave early).

In [ ]:
# Print median values side by side
print('Median values by Churn status:')
df_raw.groupby('Churn')[NUM_COLS].median().round(2)

## 5. Categorical Features

In [ ]:
# Show unique values for each categorical column
print('Unique values per categorical column:')
for col in CAT_COLS:
    vals = df_raw[col].unique().tolist()
    print(f'  {col:20s}: {vals}')

In [ ]:
# Churn rate for each category within each feature
def churn_rate_by_feature(df, col):
    return (
        df.groupby(col)['Churn_binary']
        .agg(['mean', 'count'])
        .rename(columns={'mean': 'churn_rate', 'count': 'n_customers'})
        .assign(churn_rate=lambda x: x['churn_rate'] * 100)
        .sort_values('churn_rate', ascending=False)
    )

# Quick look at contract type — usually the most important feature
print('Churn rate by Contract type:')
churn_rate_by_feature(df_raw, 'Contract')

In [ ]:
# Plot churn rate for key categorical features
KEY_CATS = ['Contract', 'InternetService', 'PaymentMethod', 'tenure_group_plot']

# Create tenure groups for plotting
df_plot = df_raw.copy()
df_plot['tenure_group_plot'] = pd.cut(
    df_plot['tenure'],
    bins=[0, 12, 36, 72],
    labels=['0-12 months', '13-36 months', '37+ months']
)

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(KEY_CATS):
    rates = churn_rate_by_feature(df_plot, col).reset_index()
    bars = axes[i].barh(
        rates[col].astype(str),
        rates['churn_rate'],
        color=sns.color_palette('muted', len(rates))
    )
    # Add percentage labels
    for bar, rate in zip(bars, rates['churn_rate']):
        axes[i].text(
            bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
            f'{rate:.1f}%', va='center', fontsize=10
        )
    axes[i].set_xlabel('Churn Rate (%)')
    axes[i].set_title(f'Churn Rate by {col.replace("_plot", "")}')
    axes[i].set_xlim(0, rates['churn_rate'].max() * 1.25)

plt.suptitle('Churn Rate by Key Categorical Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/04_churn_rate_by_category.png', dpi=150)
plt.show()

## 6. Churn Rate by All Features (Heatmap Overview)

In [ ]:
# Binary and simple Yes/No features — compare churn rates side by side
binary_features = [
    'gender', 'Partner', 'Dependents', 'PhoneService',
    'PaperlessBilling', 'SeniorCitizen'
]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(binary_features):
    col_str = df_raw[col].astype(str)  # SeniorCitizen is int, convert for groupby
    rates = df_raw.assign(**{col: col_str}).groupby(col)['Churn_binary'].mean() * 100

    axes[i].bar(rates.index, rates.values,
                color=[CHURN_PALETTE.get(k, '#999999') for k in rates.index])
    axes[i].set_title(f'Churn Rate by {col}')
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_ylim(0, 55)
    for j, (label, val) in enumerate(zip(rates.index, rates.values)):
        axes[i].text(j, val + 0.8, f'{val:.1f}%', ha='center', fontsize=10)

plt.suptitle('Churn Rate — Binary Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/05_churn_rate_binary.png', dpi=150)
plt.show()

In [ ]:
# Service features — compare churn rates
service_features = [
    'MultipleLines', 'OnlineSecurity', 'OnlineBackup',
    'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies'
]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for i, col in enumerate(service_features):
    rates = df_raw.groupby(col)['Churn_binary'].mean().sort_values(ascending=False) * 100
    colors = sns.color_palette('coolwarm', len(rates))
    axes[i].bar(rates.index, rates.values, color=colors)
    axes[i].set_title(col)
    axes[i].set_ylabel('Churn Rate (%)')
    axes[i].set_ylim(0, 60)
    axes[i].tick_params(axis='x', rotation=20)
    for j, val in enumerate(rates.values):
        axes[i].text(j, val + 0.8, f'{val:.1f}%', ha='center', fontsize=9)

# Hide unused subplot
axes[-1].set_visible(False)

plt.suptitle('Churn Rate — Service Features', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/06_churn_rate_services.png', dpi=150)
plt.show()

# Customers WITHOUT OnlineSecurity and TechSupport churn much more.
# This is a strong signal — these features will be important in the model.

## 7. Correlation Analysis

In [ ]:
# Point-biserial correlation: numeric feature vs binary churn
from scipy import stats

correlations = {}
for col in NUM_COLS:
    corr, pval = stats.pointbiserialr(df_raw['Churn_binary'], df_raw[col])
    correlations[col] = {'correlation': round(corr, 4), 'p_value': round(pval, 6)}

print('Point-biserial correlation with Churn:')
pd.DataFrame(correlations).T.sort_values('correlation')

In [ ]:
# Correlation between numeric features (check for multicollinearity)
corr_matrix = df_raw[NUM_COLS + ['Churn_binary']].corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='RdBu_r',
    center=0,
    vmin=-1, vmax=1,
    linewidths=0.5,
    ax=ax
)
ax.set_title('Correlation Matrix — Numeric Features + Churn', fontsize=12)
plt.tight_layout()
plt.savefig('../reports/figures/07_correlation_heatmap.png', dpi=150)
plt.show()

# TotalCharges and tenure are highly correlated (0.83).
# MonthlyCharges and TotalCharges are also correlated.
# We will address this with our derived feature 'charge_per_service'.

In [ ]:
# Pair plot to see relationships between numeric features split by churn
pair_df = df_raw[NUM_COLS + ['Churn']].copy()

pair_grid = sns.pairplot(
    pair_df,
    hue='Churn',
    palette=CHURN_PALETTE,
    plot_kws={'alpha': 0.4, 's': 15},
    diag_kind='kde'
)
pair_grid.figure.suptitle('Pair Plot — Numeric Features by Churn', y=1.02, fontsize=12)
plt.savefig('../reports/figures/08_pairplot.png', dpi=120, bbox_inches='tight')
plt.show()

## 8. Feature Engineering Preview

In [ ]:
# Preview the derived features we will build in the pipeline
df_fe = df_raw.copy()

# Tenure groups
df_fe['tenure_group'] = pd.cut(
    df_fe['tenure'],
    bins=[0, 12, 36, 72],
    labels=['new', 'mid', 'long']
)

# Total number of services subscribed
service_cols = [
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies'
]
df_fe['total_services'] = (df_fe[service_cols] == 'Yes').sum(axis=1)

# Monthly cost per service
df_fe['charge_per_service'] = df_fe['MonthlyCharges'] / (df_fe['total_services'] + 1)

# Churn rate by tenure group
print('Churn rate by tenure group:')
print(churn_rate_by_feature(df_fe, 'tenure_group'))

In [ ]:
# Total services vs churn — strong signal?
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Distribution of total services by churn
sns.histplot(
    data=df_fe, x='total_services', hue='Churn',
    palette=CHURN_PALETTE, multiple='dodge',
    discrete=True, ax=axes[0]
)
axes[0].set_title('Total Services by Churn')
axes[0].set_xlabel('Number of services')

# Churn rate per number of services
svc_rates = df_fe.groupby('total_services')['Churn_binary'].mean() * 100
axes[1].bar(svc_rates.index, svc_rates.values, color='steelblue')
axes[1].set_title('Churn Rate by Number of Services')
axes[1].set_xlabel('Number of services')
axes[1].set_ylabel('Churn Rate (%)')
for x, val in zip(svc_rates.index, svc_rates.values):
    axes[1].text(x, val + 0.5, f'{val:.1f}%', ha='center', fontsize=9)

plt.suptitle('Derived Feature: total_services', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/figures/09_total_services_churn.png', dpi=150)
plt.show()

# Customers with fewer services churn more — total_services will be a useful feature.

## 9. Key Findings Summary

In [ ]:
overall_churn = df_raw['Churn_binary'].mean() * 100

# Build a summary table with the most important churn signals
findings = [
    ('Overall churn rate', f'{overall_churn:.1f}%', '—'),
    ('Month-to-month contract', f"{churn_rate_by_feature(df_raw, 'Contract').loc['Month-to-month', 'churn_rate']:.1f}%", 'High risk'),
    ('Two year contract',       f"{churn_rate_by_feature(df_raw, 'Contract').loc['Two year', 'churn_rate']:.1f}%",       'Low risk'),
    ('No OnlineSecurity',       f"{df_raw[df_raw['OnlineSecurity']=='No']['Churn_binary'].mean()*100:.1f}%",             'High risk'),
    ('Has OnlineSecurity',      f"{df_raw[df_raw['OnlineSecurity']=='Yes']['Churn_binary'].mean()*100:.1f}%",            'Low risk'),
    ('Fiber optic internet',    f"{df_raw[df_raw['InternetService']=='Fiber optic']['Churn_binary'].mean()*100:.1f}%",   'High risk'),
    ('Tenure < 12 months',      f"{df_raw[df_raw['tenure']<=12]['Churn_binary'].mean()*100:.1f}%",                      'High risk'),
    ('Tenure > 36 months',      f"{df_raw[df_raw['tenure']>36]['Churn_binary'].mean()*100:.1f}%",                       'Low risk'),
    ('Electronic check payment',f"{df_raw[df_raw['PaymentMethod']=='Electronic check']['Churn_binary'].mean()*100:.1f}%",'High risk'),
    ('Senior citizen',          f"{df_raw[df_raw['SeniorCitizen']==1]['Churn_binary'].mean()*100:.1f}%",                'High risk'),
]

summary = pd.DataFrame(findings, columns=['Feature / Segment', 'Churn Rate', 'Signal'])
print('\n=== KEY EDA FINDINGS ===')
print(summary.to_string(index=False))

In [ ]:
print("""
=== SUMMARY FOR FEATURE ENGINEERING (next notebook) ===

DATA QUALITY:
  - TotalCharges had 11 hidden blanks for tenure=0 customers → filled with 0.
  - No duplicates. Dataset is clean.

CLASS IMBALANCE:
  - 73.5% No Churn vs 26.5% Churn.
  - Use scale_pos_weight in XGBoost. Report AUC-PR, not accuracy.

TOP CHURN SIGNALS:
  1. Contract type — month-to-month customers churn ~3x more than 2-year contracts.
  2. Tenure — new customers (< 12 months) are the highest risk group.
  3. Internet service — Fiber optic users churn most (possibly price-sensitive).
  4. No security/support services — customers without protection churn more.
  5. Electronic check payment — highest churn among payment methods.

MULTICOLLINEARITY:
  - TotalCharges and tenure are highly correlated (r=0.83).
  - MonthlyCharges and TotalCharges also correlated.
  - Derived feature 'charge_per_service' will capture a cleaner signal.

FEATURES TO ENGINEER:
  - tenure_group (new / mid / long)
  - total_services (count of subscribed services)
  - charge_per_service (MonthlyCharges / total_services + 1)
""")